In [4]:
import pandas as pd
import statsmodels.api as sm

In [5]:
# Load dataset
df = pd.read_csv('FE-GWP1_model_selection_2.csv')

In [6]:
# Target variable
#y = df["Y"]

In [7]:
# Predictors
predictors = ["Z1 ", "Z2", "Z3", "Z4", "Z5"]

In [8]:
def fit_model(data, variables):
    """
    Fit an OLS regression model using the specified predictors.
    """
    #X = sm.add_constant(data[variables])
    #return sm.OLS(y, X).fit()
    X = sm.add_constant(data[variables])
    y = data["Y"]

    return sm.OLS(y, X).fit()

In [9]:
def backward_adjusted_r2(data, predictors):
    """
    Backward elimination using adjusted R-squared.
    At each step, remove the variable whose removal produces
    the highest adjusted R-squared. Stop when removing a variable
    no longer improves adjusted R-squared.
    """
    selected = predictors.copy()
    history = []

    current_model = fit_model(data, selected)
    current_adj_r2 = current_model.rsquared_adj

    history.append({
        "step": 0,
        "removed": None,
        "variables": selected.copy(),
        "adjusted_R2": current_adj_r2
    })

    step = 1

    while len(selected) > 1:

        candidates = []

        for variable in selected:
            remaining = [x for x in selected if x != variable]
            model = fit_model(data, remaining)

            candidates.append({
                "removed": variable,
                "variables": remaining,
                "adjusted_R2": model.rsquared_adj
            })

        # Find the removal giving the highest adjusted R-squared
        best = max(candidates, key=lambda x: x["adjusted_R2"])

        if best["adjusted_R2"] > current_adj_r2:

            selected = best["variables"]
            current_adj_r2 = best["adjusted_R2"]

            history.append({
                "step": step,
                "removed": best["removed"],
                "variables": selected.copy(),
                "adjusted_R2": current_adj_r2
            })

            step += 1

        else:
            break

    final_model = fit_model(data, selected)

    return selected, final_model, history

In [10]:
backward_adjusted_r2(df, predictors)

(['Z1 ', 'Z2', 'Z3', 'Z4', 'Z5'],
 [{'step': 0,
   'removed': None,
   'variables': ['Z1 ', 'Z2', 'Z3', 'Z4', 'Z5'],
   'adjusted_R2': np.float64(0.993573339252398)}])

In [10]:
print(df.columns)
print(predictors)

Index(['Y', 'Z1 ', 'Z2', 'Z3', 'Z4', 'Z5'], dtype='object')
['Z1', 'Z2', 'Z3', 'Z4', 'Z5']


In [21]:
print(df.shape)
print(df.columns.tolist())
print(df.head())

(100, 6)
['Y', 'Z1 ', 'Z2', 'Z3', 'Z4', 'Z5']
          Y       Z1         Z2        Z3        Z4        Z5
0  2.172296  0.121634 -0.051562  0.570616  1.279931  0.075233
1  0.502380  0.025446 -0.093062  0.304875 -0.582292  0.377388
2  0.711362 -0.136716 -0.082229 -0.191680 -0.647970  1.230986
3 -0.557168 -0.284459 -0.170922 -0.853670 -1.256146 -0.991686
4  1.500199  0.105205 -0.169141  0.826558  0.640945  1.099873


In [27]:
selected_adj_r2, model_adj_r2, history_adj_r2 = backward_adjusted_r2(
    df, predictors
)
print("Selected variables:", selected_adj_r2)
print("Adjusted R-squared:", model_adj_r2.rsquared_adj)

Selected variables: ['Z1 ', 'Z2', 'Z3', 'Z4', 'Z5']
Adjusted R-squared: 0.993573339252398


In [18]:
selected, final_model, history = backward_adjusted_r2(
    df, predictors
)

print(final_model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.994
Model:                            OLS   Adj. R-squared:                  0.994
Method:                 Least Squares   F-statistic:                     3062.
Date:                Fri, 18 Sep 2026   Prob (F-statistic):          2.07e-102
Time:                        12:08:38   Log-Likelihood:                 88.951
No. Observations:                 100   AIC:                            -165.9
Df Residuals:                      94   BIC:                            -150.3
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.0097      0.013     77.496      0.0

In [20]:
def backward_aic(data, predictors):
    """
    Backward elimination using AIC.

    At each step, remove the variable whose removal produces
    the lowest AIC. Stop when no removal decreases AIC.
    """
    selected = predictors.copy()
    history = []

    current_model = fit_model(data, selected)
    current_aic = current_model.aic

    history.append({
        "step": 0,
        "removed": None,
        "variables": selected.copy(),
        "AIC": current_aic
    })

    step = 1

    while len(selected) > 1:

        candidates = []

        for variable in selected:
            remaining = [x for x in selected if x != variable]
            model = fit_model(data, remaining)

            candidates.append({
                "removed": variable,
                "variables": remaining,
                "AIC": model.aic
            })

        # Find the removal giving the lowest AIC
        best = min(candidates, key=lambda x: x["AIC"])

        if best["AIC"] < current_aic:

            selected = best["variables"]
            current_aic = best["AIC"]

            history.append({
                "step": step,
                "removed": best["removed"],
                "variables": selected.copy(),
                "AIC": current_aic
            })

            step += 1

        else:
            break

    final_model = fit_model(data, selected)

    return selected, final_model, history

In [21]:
backward_aic(df, predictors)

(['Z1 ', 'Z2', 'Z3', 'Z4', 'Z5'],
 [{'step': 0,
   'removed': None,
   'variables': ['Z1 ', 'Z2', 'Z3', 'Z4', 'Z5'],
   'AIC': np.float64(-165.9022483376799)}])

In [22]:
selected, final_model, history = backward_aic(
    df, predictors
)

print(final_model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.994
Model:                            OLS   Adj. R-squared:                  0.994
Method:                 Least Squares   F-statistic:                     3062.
Date:                Fri, 18 Sep 2026   Prob (F-statistic):          2.07e-102
Time:                        12:10:52   Log-Likelihood:                 88.951
No. Observations:                 100   AIC:                            -165.9
Df Residuals:                      94   BIC:                            -150.3
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.0097      0.013     77.496      0.0

In [31]:
def backward_bic(data, predictors):
    """
    Backward elimination using BIC.

    At each step, remove the variable whose removal produces
    the lowest BIC. Stop when no removal decreases BIC.
    """
    selected = predictors.copy()
    history = []

    current_model = fit_model(data, selected)
    current_bic = current_model.bic

    history.append({
        "step": 0,
        "removed": None,
        "variables": selected.copy(),
        "BIC": current_bic
    })

    step = 1

    while len(selected) > 1:

        candidates = []

        for variable in selected:
            remaining = [x for x in selected if x != variable]
            model = fit_model(data, remaining)

            candidates.append({
                "removed": variable,
                "variables": remaining,
                "BIC": model.bic
            })

        # Find the removal giving the lowest BIC
        best = min(candidates, key=lambda x: x["BIC"])

        if best["BIC"] < current_bic:

            selected = best["variables"]
            current_bic = best["BIC"]

            history.append({
                "step": step,
                "removed": best["removed"],
                "variables": selected.copy(),
                "BIC": current_bic
            })

            step += 1

        else:
            break

    final_model = fit_model(data, selected)

    return selected, final_model, history

In [32]:
backward_bic(df, predictors)

(['Z1 ', 'Z2', 'Z3', 'Z4', 'Z5'],
 [{'step': 0,
   'removed': None,
   'variables': ['Z1 ', 'Z2', 'Z3', 'Z4', 'Z5'],
   'BIC': np.float64(-150.27122722175136)}])

In [33]:
# 1. Adjusted R-squared
adj_r2_vars, adj_r2_model, adj_r2_history = backward_adjusted_r2(
    df, predictors
)

# 2. AIC
aic_vars, aic_model, aic_history = backward_aic(
    df, predictors
)

# 3. BIC
bic_vars, bic_model, bic_history = backward_bic(
    df, predictors
)

In [34]:
print("ADJUSTED R-SQUARED SELECTION")
for result in adj_r2_history:
    print(result)

print("\nAIC SELECTION")
for result in aic_history:
    print(result)

print("\nBIC SELECTION")
for result in bic_history:
    print(result)

ADJUSTED R-SQUARED SELECTION
{'step': 0, 'removed': None, 'variables': ['Z1 ', 'Z2', 'Z3', 'Z4', 'Z5'], 'adjusted_R2': np.float64(0.993573339252398)}

AIC SELECTION
{'step': 0, 'removed': None, 'variables': ['Z1 ', 'Z2', 'Z3', 'Z4', 'Z5'], 'AIC': np.float64(-165.9022483376799)}

BIC SELECTION
{'step': 0, 'removed': None, 'variables': ['Z1 ', 'Z2', 'Z3', 'Z4', 'Z5'], 'BIC': np.float64(-150.27122722175136)}


In [35]:
comparison = pd.DataFrame({
    "Method": [
        "Adjusted R-squared",
        "AIC",
        "BIC"
    ],
    "Selected Predictors": [
        ", ".join(adj_r2_vars),
        ", ".join(aic_vars),
        ", ".join(bic_vars)
    ],
    "Adjusted R-squared": [
        adj_r2_model.rsquared_adj,
        aic_model.rsquared_adj,
        bic_model.rsquared_adj
    ],
    "AIC": [
        adj_r2_model.aic,
        aic_model.aic,
        bic_model.aic
    ],
    "BIC": [
        adj_r2_model.bic,
        aic_model.bic,
        bic_model.bic
    ]
})

comparison

,Method,Selected Predictors,Adjusted R-squared,AIC,BIC
0,Adjusted R-squared,"Z1 , Z2, Z3, Z4, Z5",0.993573,-165.902248,-150.271227
1,AIC,"Z1 , Z2, Z3, Z4, Z5",0.993573,-165.902248,-150.271227
2,BIC,"Z1 , Z2, Z3, Z4, Z5",0.993573,-165.902248,-150.271227


In [23]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X = df[["Z1 ", "Z2", "Z3", "Z4", "Z5"]]
y = df["Y"]

lasso = make_pipeline(
    StandardScaler(),
    LassoCV(cv=10, random_state=42)
)

lasso.fit(X, y)

model = lasso.named_steps["lassocv"]

print("Best alpha:", model.alpha_)

for variable, coefficient in zip(X.columns, model.coef_):
    print(variable, coefficient)

Best alpha: 0.0011785282327137123
Z1  0.07948151672689792
Z2 0.11092537131729813
Z3 -0.4071753989569318
Z4 1.1709853633412848
Z5 0.20997375940473095


In [27]:
# Coefficients
coefficients = pd.Series(
    model.coef_,
    index=X.columns
)

print("LASSO MODEL SUMMARY")
print("=" * 40)
print(f"Best alpha: {model.alpha_:.6f}")
print(f"R-squared: {lasso.score(X, y):.6f}")
print(f"Number of selected variables: {(model.coef_ != 0).sum()}")

print("\nCoefficients:")
print(coefficients)

LASSO MODEL SUMMARY
Best alpha: 0.001179
R-squared: 0.993893
Number of selected variables: 5

Coefficients:
Z1     0.079482
Z2     0.110925
Z3    -0.407175
Z4     1.170985
Z5     0.209974
dtype: float64
